# ✅ Solution: AdaLN-Zero

Implement a **simplified AdaLN-Zero residual branch**, with zero initialization that still allows learning:

`branch(x, cond) = gate(cond) * ((1 + scale(cond)) * LN(x) + shift(cond))`

The layer returns the **branch**, not `x + branch`. Its output starts at zero; a caller adding the residual (`x + branch`) therefore starts as identity. The `1 + scale` matters: multiplying zero gate by a zero branch would give dead gradients.

### Signature
```python
class AdaLNZero(nn.Module):
    def __init__(self, hidden_dim: int, cond_dim: int):
        ...
    def forward(self, x: torch.Tensor, cond: torch.Tensor) -> torch.Tensor:
        ...
```

### Requirements
- x is `(B, S, hidden_dim)`, cond is `(B, cond_dim)`; return the same shape as x.
- LN normalizes the last dimension with population variance and `eps=1e-5`, without fixed affine parameters.
- Predict scale and shift with learned conditioning projections; zero-initialize their final weights and biases.
- Expose **`self.gate_proj = nn.Linear(cond_dim, hidden_dim)`**, with zero weights and bias, and use its output as the gate. This public name lets tests open/close the gate independently. All other internal names are unrestricted.
- Broadcast per-sample scale, shift, and gate across tokens. Use `1 + scale`, so opening only the gate at initialization reveals `LN(x)`.
- Preserve autograd; the gate must receive a nonzero gradient on a nontrivial training example.

A full DiT block also includes attention/MLP transformations inside its gated residual branches; this exercise isolates normalization, modulation, and gating.

In [ ]:
import torch
import torch.nn as nn

In [ ]:
# ✅ SOLUTION

class AdaLNZero(nn.Module):
    def __init__(self, hidden_dim: int, cond_dim: int):
        super().__init__()
        self.norm = nn.LayerNorm(hidden_dim, elementwise_affine=False)
        
        # MLP for scale offsets and shifts; the separate gate_proj is used below
        self.cond_mlp = nn.Sequential(
            nn.SiLU(),
            nn.Linear(cond_dim, 2 * hidden_dim)  # scale, shift
        )
        
        # Zero-initialize the final projection
        nn.init.zeros_(self.cond_mlp[1].weight)
        nn.init.zeros_(self.cond_mlp[1].bias)
        
        self.gate_proj = nn.Linear(cond_dim, hidden_dim)
        nn.init.zeros_(self.gate_proj.weight)
        nn.init.zeros_(self.gate_proj.bias)
    
    def forward(self, x: torch.Tensor, cond: torch.Tensor) -> torch.Tensor:
        x_norm = self.norm(x)
        gamma, beta = self.cond_mlp(cond).chunk(2, dim=-1)
        gate = self.gate_proj(cond)
        
        gamma = gamma.unsqueeze(1)
        beta = beta.unsqueeze(1)
        gate = gate.unsqueeze(1)
        
        return gate * ((1 + gamma) * x_norm + beta)

In [ ]:
# Verify
adaln = AdaLNZero(hidden_dim=64, cond_dim=128)
x = torch.randn(2, 10, 64)
cond = torch.randn(2, 128)
out = adaln(x, cond)
print(f"Output max (should be ~0): {out.abs().max().item():.6f}")

In [ ]:
from torch_judge import check
check("adaln_zero")